# LLM Prompt Runner

Run prompts on different LLM models (Colab AI, OpenAI). Experiments and prompts are defined in `start_here/experiments/definition.py` and stored in the database; the notebook loads them and writes results back to the DB.

In [ ]:
# (Optional) Install dependencies if needed (run once, e.g. in Colab)
# You can comment this out locally if packages are already installed.
%pip install openai psycopg2-binary -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
from pathlib import Path

# Dynamically locate the project root (directory that has both
# `integrations` and `managers`) starting from the current working dir
_cwd = Path.cwd().resolve()
_project_root = None
for candidate in [_cwd, *_cwd.parents]:
    if (candidate / "integrations").exists() and (candidate / "managers").exists():
        _project_root = candidate
        break

print("CWD:", _cwd)
print("Project root:", _project_root)

if _project_root is None:
    raise RuntimeError(
        f"Could not locate project root starting from {_cwd}. "
        "Make sure you run this notebook from inside the cloned repository."
    )

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
    print("Added to sys.path:", _project_root)

from datetime import datetime
from integrations.colab_llm_provider import ColabLLMProvider
from integrations.openai_llm_provider import OpenAILLMProvider
from managers.prompt_output_manager import save_results
from database.experiment_log import (
    list_experiments,
    get_experiment,
    get_prompts,
)
from start_here.experiments.constants import (
    MODEL_PROVIDERS,
    MODEL_KEYS,
    OPENAI_API_KEY,
)
import os
_ = os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY or "")

CWD: /home/marcelo/Doutorado/llm-code-review-on-smells/start_here/notebooks
Project root: /home/marcelo/Doutorado/llm-code-review-on-smells
Added to sys.path: /home/marcelo/Doutorado/llm-code-review-on-smells


'<redacted>'

## Execution

### 0. Create tables and seed from experiments.py (run once)

Creates `experiment_setup`, `prompt`, and `llm_prompt_results` if they do not exist, then inserts experiment definitions from `start_here/experiments/definition.py`. Existing experiment names are skipped.

In [2]:
from managers.database_manager import load_database
from importlib import reload

from start_here.experiments import constants
import database.experiment_log as experiment_log
import managers.database_manager as db_manager

reload(constants)
reload(experiment_log)
reload(db_manager)

print("DB name    :", constants.RESULTS_DATABASE_NAME)
print("DB host    :", constants.RESULTS_DATABASE_HOST)
print("DB port    :", constants.RESULTS_DATABASE_PORT)
print("DB user    :", constants.RESULTS_DATABASE_USER)
print("DB password:", constants.RESULTS_DATABASE_PASSWORD)

load_database()

DB name    : datasets_db
DB host    : localhost
DB port    : 5432
DB user    : postgres
DB password: postgres


Experiment 'test_gemini_flash' already exists, skipping.
Experiment 'test_from_file' already exists, skipping.
Experiment 'test_openai_gpt_4_1' already exists, skipping.


### Available models

In [3]:
print("Available models (use MODEL_KEYS[i] in step 1):")
for i, k in enumerate(MODEL_KEYS):
    print(f"  [{i}] {k} ({MODEL_PROVIDERS[k]})")

Available models (use MODEL_KEYS[i] in step 1):
  [0] colab_gemini-2.0-flash (colab)
  [1] colab_gemini-2.0-flash-lite (colab)
  [2] colab_gemini-1.5-pro (colab)
  [3] openai_gpt-4.1 (openai)
  [4] openai_gpt-4 (openai)
  [5] openai_gpt-4-turbo (openai)
  [6] openai_gpt-3.5-turbo (openai)


### 1. Choose or create experiment (model + decoding from DB)

In [4]:
experiments = list_experiments()
for exp in experiments:
    print(f"  id={exp['id']} | {exp['name']} | {exp['model_key']} | t={exp['temperature']} top_p={exp['top_p']} max_tokens={exp['max_tokens']}")
if not experiments:
    print("  No experiments yet. Define them in start_here/experiments/definition.py and run load_database().")

  id=3 | test_openai_gpt_4_1 | openai_gpt-4.1 | t=0.0 top_p=1.0 max_tokens=2048
  id=2 | test_from_file | gemini-2.0-flash | t=0.0 top_p=1.0 max_tokens=2048
  id=1 | test_gemini_flash | gemini-2.0-flash | t=0.0 top_p=1.0 max_tokens=2048


In [5]:
EXPERIMENT_ID = 3  # use an existing id from list_experiments()

experiment = get_experiment(EXPERIMENT_ID)
if experiment:
    print(f"Experiment: {experiment['name']} (id={EXPERIMENT_ID})")
    print(f"Model: {experiment['model_key']} | temperature={experiment['temperature']} | top_p={experiment['top_p']} | max_tokens={experiment['max_tokens']}")
else:
    print("No experiment with that id. Define experiments in start_here/experiments/definition.py and run load_database().")

Experiment: test_openai_gpt_4_1 (id=3)
Model: openai_gpt-4.1 | temperature=0.0 | top_p=1.0 | max_tokens=2048


### 3. Run code-review prompt over `mlcq_files`

Para cada arquivo elegível em `mlcq_files` (que tenha pelo menos uma occurrence em `(target_smells × {major, critical})`), faz **1 chamada de LLM** com o `file_content` injetado em `prompts/code-review-prompt-v04.txt` (substituindo `<code file>`), e grava em `llm_prompt_results` com `mlcq_file_id = mlcq_files.id`.

Cada arquivo gera 1 response — independente de quantos smells ele tenha. A expansão para múltiplas tasks de rotulagem (uma por smell-alvo) acontece depois, em [run_human_evaluation.ipynb](./run_human_evaluation.ipynb).

Estrutura abaixo:
- **3.0** — Helper `run_llm_batch(demo_only, file_limit=None)`
- **3.a** — Piloto (`demo_only=True`)
- **3.b** — Rotulagem completa (`demo_only=False`)

In [6]:
def _get_llm(model_key: str):
    if model_key not in MODEL_PROVIDERS:
        raise ValueError(f"Model '{model_key}' not found. Available: {list(MODEL_PROVIDERS.keys())}")
    provider = MODEL_PROVIDERS[model_key]
    if provider == "colab":
        return ColabLLMProvider(model_key)
    if provider == "openai":
        return OpenAILLMProvider(model_key)
    raise ValueError(f"Unsupported provider: {provider}")


### 3.0 Helper `run_llm_batch`

Define a função usada pelas células 3.a e 3.b. Roda esta célula uma vez antes de executar qualquer dos lotes.

In [7]:
from start_here.experiments.constants import (
    RESULTS_DATABASE_NAME,
    RESULTS_DATABASE_HOST,
    RESULTS_DATABASE_PORT,
    RESULTS_DATABASE_USER,
    RESULTS_DATABASE_PASSWORD,
)
from managers.prompt_input_manager import read_from_file
from database.experiment_log import (
    add_prompt,
    EVALUATION_TARGET_SEVERITIES,
    EVALUATION_TARGET_SMELLS,
)
import psycopg2


def run_llm_batch(demo_only, file_limit=None,
                  target_smells=EVALUATION_TARGET_SMELLS,
                  target_severities=EVALUATION_TARGET_SEVERITIES):
    """Roda o LLM sobre arquivos de mlcq_files filtrados por demo_set.

    Args:
        demo_only: True = só demo_set; False = só NÃO demo_set; None = todos.
        file_limit: máximo de arquivos no lote (None = sem limite).
        target_smells / target_severities: filtros de elegibilidade do arquivo.
    """
    base_prompt_text = read_from_file("code-review-prompt-v04.txt")[0]

    if demo_only is True:
        demo_filter_sql = "AND f.demo_set = TRUE"
    elif demo_only is False:
        demo_filter_sql = "AND f.demo_set = FALSE"
    else:
        demo_filter_sql = ""

    connection = psycopg2.connect(
        dbname=RESULTS_DATABASE_NAME,
        user=RESULTS_DATABASE_USER,
        password=RESULTS_DATABASE_PASSWORD,
        host=RESULTS_DATABASE_HOST,
        port=RESULTS_DATABASE_PORT,
    )

    try:
        with connection.cursor() as cursor:
            cursor.execute(
                f"""
                SELECT f.id, f.file_content
                FROM mlcq_files f
                WHERE EXISTS (
                    SELECT 1 FROM mlcq_smell_occurrences o
                    WHERE o.id = ANY(f.smell_occurrence_ids)
                      AND LOWER(o.smell) IN %s
                      AND o.severity IN %s
                )
                {demo_filter_sql}
                ORDER BY f.id
                """ + ("LIMIT %s" if file_limit is not None else ""),
                tuple(
                    [target_smells, target_severities]
                    + ([file_limit] if file_limit is not None else [])
                ),
            )
            files_to_process = cursor.fetchall()

        print(
            f"demo_only={demo_only}  ·  arquivos a processar: {len(files_to_process)}  "
            f"·  modelo: {experiment['model_key']}"
        )

        llm = _get_llm(experiment["model_key"])
        results = []

        for mlcq_file_id, file_content in files_to_process:
            if not file_content:
                continue

            prompt_text = base_prompt_text.replace("<code file>", file_content)
            prompt_id = add_prompt(EXPERIMENT_ID, prompt_text)

            start_time = datetime.now()
            try:
                response = llm.generate(
                    prompt_text,
                    temperature=experiment["temperature"],
                    top_p=experiment["top_p"],
                    max_tokens=experiment["max_tokens"],
                )
                duration = (datetime.now() - start_time).total_seconds()
                result = {
                    "prompt": prompt_text,
                    "response": response,
                    "model_key": experiment["model_key"],
                    "model_name": llm.model_name,
                    "timestamp": start_time.isoformat(),
                    "duration_seconds": duration,
                    "success": True,
                    "prompt_id": prompt_id,
                    "mlcq_file_id": mlcq_file_id,
                    "error": None,
                }
            except Exception as e:
                duration = (datetime.now() - start_time).total_seconds()
                result = {
                    "prompt": prompt_text,
                    "response": None,
                    "model_key": experiment["model_key"],
                    "model_name": llm.model_name,
                    "timestamp": start_time.isoformat(),
                    "duration_seconds": duration,
                    "success": False,
                    "prompt_id": prompt_id,
                    "mlcq_file_id": mlcq_file_id,
                    "error": str(e),
                }

            results.append(result)

        if results:
            save_results(EXPERIMENT_ID, results, experiment["model_key"], llm.model_name)
        return len(results)
    finally:
        connection.close()


### 3.a Piloto (`demo_only=True`)

Roda o LLM **só nos arquivos do `demo_set`** (8 arquivos estratificados, ver `datasets/select_demo_set.py`). Use para a fase de piloto/codebook (Seção 7.3 do manual de rotulagem).

Repita esta célula trocando `EXPERIMENT_ID` na célula anterior pra rodar contra cada modelo que quer comparar.

In [8]:
run_llm_batch(demo_only=True, file_limit=None)

demo_only=True  ·  arquivos a processar: 8  ·  modelo: openai_gpt-4.1

[2026-04-28T15:11:08.068059] Results saved to database: postgresql://localhost:5432/datasets_db (experiment_id=3, model_key=openai_gpt-4.1, total_rows=8)


8

### 3.b Rotulagem completa (`demo_only=False`)

Roda o LLM nos arquivos **fora** do `demo_set`. Ajuste `file_limit` conforme o orçamento de chamadas de LLM (cada arquivo = 1 chamada).

Use só **depois** que o piloto validou o codebook (Cohen's Kappa OK).

In [8]:
run_llm_batch(demo_only=False, file_limit=None)   # ajuste file_limit conforme orçamento; None = todos

demo_only=False  ·  arquivos a processar: 428  ·  modelo: openai_gpt-4.1

[2026-05-01T19:00:32.279458] Results saved to database: postgresql://localhost:5432/datasets_db (experiment_id=3, model_key=openai_gpt-4.1, total_rows=28)


428